In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d nibinv23/iam-handwriting-word-database

In [ ]:
!unzip iam-handwriting-word-database.zip -d data

In [ ]:
import os
image_paths=[]
labels=[]
with open('data/words_new.txt') as f:
    for line in f:
        if line.startswith('#'): continue
        parts=line.strip().split()
        if len(parts)<9: continue
        word=parts[-1]
        fid=parts[0]
        f1=fid.split('-')[0]
        f2='-'.join(fid.split('-')[:2])
        path=f'data/iam_words/words/{f1}/{f2}/{fid}.png'
        if os.path.exists(path):
            image_paths.append(path)
            labels.append(word)
print(len(image_paths))

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(image_paths,labels,test_size=0.2,random_state=42)

# Filter out non-existent or empty image paths after splitting
import os
def filter_existent_paths(paths, corresponding_labels):
    existent_paths = []
    existent_labels = []
    for p, l in zip(paths, corresponding_labels):
        # Check if the file exists and is not empty
        if os.path.exists(p) and os.path.getsize(p) > 0:
            existent_paths.append(p)
            existent_labels.append(l)
    return existent_paths, existent_labels

x_train, y_train = filter_existent_paths(x_train, y_train)
x_test, y_test = filter_existent_paths(x_test, y_test)

print(f"Number of existent training images: {len(x_train)}")
print(f"Number of existent testing images: {len(x_test)}")

In [ ]:
all_text=''.join(y_train)
vocab=sorted(set(all_text))
char_to_num={c:i for i,c in enumerate(vocab)}
num_to_char={i:c for c,i in char_to_num.items()}

def encode(t): return [char_to_num[c] for c in t]
y_train_seq=[encode(t) for t in y_train]
y_test_seq=[encode(t) for t in y_test]
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len=max(len(x) for x in y_train_seq)
y_train_seq=pad_sequences(y_train_seq,maxlen=max_len,padding='post')
y_test_seq=pad_sequences(y_test_seq,maxlen=max_len,padding='post')

In [ ]:
import tensorflow as tf

IMG_HEIGHT = 32
IMG_WIDTH = 128

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

    # Normalize
    img = img / 255.0

    # 🔥 Data Augmentation (ONLY for training)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.6, 1.4)

    return img, label

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train_seq))
train_ds = train_ds.map(load_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(32).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test_seq))
test_ds = test_ds.map(load_image, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(32).prefetch(AUTOTUNE)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
model = tf.keras.Sequential([
    tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(256, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Resizing(IMG_HEIGHT//8, 19),
    layers.Reshape((19, -1)),
    layers.Bidirectional(layers.LSTM(256, return_sequences=True)),
    layers.Dropout(0.4),
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
    layers.Dropout(0.3),
    layers.Dense(len(vocab)+1, activation='softmax')
])

In [ ]:


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(patience=4, restore_best_weights=True)

lr_schedule = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-5
)

history = model.fit(
    train_ds,
    epochs=25,
    validation_data=test_ds,
    callbacks=[early_stop, lr_schedule]
)

In [ ]:
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc*100:.2f}%")

In [ ]:
max_val_acc = max(history.history['val_accuracy'])
print(max_val_acc)

In [ ]:
import os
import random
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
IMG_WIDTH = 128
IMG_HEIGHT = 32
BASE_PATH = "data/iam_words/words"

# =========================
# LOAD GROUND TRUTH (FIXED)
# =========================
gt_dict = {}

with open("data/iam_words/words.txt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        parts = line.strip().split()
        if len(parts) < 9:
            continue

        word_id = parts[0]   # e.g. n04-156-04-05
        word = parts[-1]

        gt_dict[word_id] = word

# =========================
# PREPROCESS (same as training)
# =========================
def preprocess_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0
    return img

# =========================
# DECODE (FIXED)
# =========================
def decode(pred):
    pred_ids = np.argmax(pred, axis=-1)

    result = []
    prev = -1

    for i in pred_ids:
        # remove duplicates + ignore blank token (last index)
        if i != prev and i != len(vocab) - 1:
            result.append(vocab[i])
        prev = i

    return "".join(result)

# =========================
# MAIN FUNCTION
# =========================
def random_result():
    # collect all images
    all_images = []
    for root, _, files in os.walk(BASE_PATH):
        for f in files:
            if f.endswith(".png"):
                all_images.append(os.path.join(root, f))

    # pick GOOD sample (avoid garbage images)
    while True:
        img_path = random.choice(all_images)
        img_id = os.path.basename(img_path).replace(".png", "")

        if img_id in gt_dict and len(gt_dict[img_id]) > 1:
            break

    gt = gt_dict[img_id]

    # preprocess
    img = preprocess_image(img_path)
    img_input = tf.expand_dims(img, axis=0)

    # predict
    pred = model.predict(img_input, verbose=0)
    pred_text = decode(pred[0])

    # display
    plt.imshow(tf.squeeze(img), cmap='gray')
    plt.title(f"GT: {gt} | Pred: {pred_text}")
    plt.axis('off')

    # print
    print("\nImage:", img_path)
    print("Ground Truth:", gt)
    print("Prediction:", pred_text)

# =========================
# RUN DEMO
# =========================
random_result()

In [ ]:
model.save("handwriting_model.h5")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model("handwriting_model.h5")

In [ ]:
model.evaluate(test_ds)

In [ ]:
height = 32
width = 128
def predict_image(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, [height, width])
    img = tf.cast(img, tf.float32) / 255.0

    img = tf.expand_dims(img, axis=0)

    pred = model.predict(img)
    pred_seq = tf.argmax(pred, axis=-1).numpy()[0]
    pred_seq = pred_seq[pred_seq != 0]
    word = ''.join([num_to_char.get(i, '') for i in pred_seq])
    return word

In [ ]:
import matplotlib.pyplot as plt

i = 0
img_path = image_paths[i]
pred = predict_image(img_path)
true_label = labels[i]
plt.imshow(plt.imread(img_path), cmap='gray')
plt.title(f"Pred: {pred} | Actual: {true_label}")
plt.axis('off')